<a href="https://colab.research.google.com/github/strGT20/Cocokin-Notebook-DS/blob/main/Cocokin_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CocokinLogo.png
## Dicoding Capstone Project
---


#### Bussines Question

1. Berdasarkan data profil kandidat tahun 2023, apa saja 5 keahlian teknis (tech skills) spesifik yang paling sering muncul sebagai missing_skills pada kandidat dengan tingkat pendidikan Sarjana (S1) yang melamar di sektor Teknologi?
2. Bagaimana perbandingan proporsi kebutuhan antara Soft Skills dan Tech Skills yang diwajibkan oleh perusahaan di sektor 'Keuangan' dibandingkan dengan sektor 'Teknologi Informasi' pada dataset lowongan pekerjaan tahun 2023?
3. Berapa rata-rata tahun pengalaman kerja yang sebenarnya diwajibkan oleh perusahaan untuk lowongan berlabel 'Entry Level' (Pemula), dan 5 sektor industri mana yang mematok standar pengalaman paling tinggi pada tahun 2023?

## 1. Setup - Install Dependencies & Download Dataset

In [ ]:
# Install dependencies
%pip install -q scipy scikit-learn seaborn matplotlib pandas numpy

In [ ]:
import os
import gdown # Menggunakan gdown karena size dataset job postings besar

# URL dataset
RESUME_URL   = "https://drive.google.com/uc?export=download&id=1KDjk4UL1CFFfF96rTgs1fjQ_luKpP-Qo"
JOBS_URL     = "https://drive.google.com/uc?export=download&id=1-R-wzAenKEdoOwnY5PJuVzCwcdSXN3eK"

# Local dir di sesi Colab
DATA_DIR   = "/content/cocokin_data"
OUTPUT_DIR = "/content/cocokin_output"
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESUME_PATH = f"{DATA_DIR}/Resume.csv"
JOBS_PATH   = f"{DATA_DIR}/job_postings.csv"

def download_if_missing(url, dest):
    # Cek apakah file sudah ada dan ukurannya logis (bukan file HTML error kecil)
    if os.path.exists(dest) and os.path.getsize(dest) > 100_000:
        print(f"  ✅ Sudah ada: {os.path.basename(dest)}")
        return

    print(f"  ⬇️  Mengunduh {os.path.basename(dest)} secara utuh (raw) ...")

    # gdown akan otomatis melewati peringatan virus file besar di Google Drive
    gdown.download(url, dest, quiet=False)

    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1_048_576
        print(f"     Selesai ({size_mb:.1f} MB)")
    else:
        print(f"     ❌ Gagal mengunduh {os.path.basename(dest)}")

download_if_missing(RESUME_URL, RESUME_PATH)
download_if_missing(JOBS_URL,   JOBS_PATH)
print("\n✅ Dataset siap digunakan.")

  ✅ Sudah ada: Resume.csv
  ✅ Sudah ada: job_postings.csv

✅ Dataset siap digunakan.


## 2. Global Imports

In [ ]:
import pandas as pd
import numpy as np
import re, ast, math, warnings
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# Tampilkan plot langsung di notebook
plt.rcParams["figure.dpi"] = 120

sns.set_theme(style="darkgrid", palette="muted")
PALETTE = [
    "#4C9BE8", "#F97316", "#22C55E", "#A855F7", "#EF4444",
    "#FACC15", "#06B6D4", "#EC4899", "#84CC16", "#F59E0B",
]

SOFT_SKILLS_MASTER = {
    'communication', 'leadership', 'analytical', 'problem solving',
    'time management', 'negotiation', 'customer service', 'teamwork',
    'interpersonal', 'adaptability', 'critical thinking', 'collaboration',
    'organization', 'creativity', 'attention to detail', 'multitasking',
    'conflict resolution', 'presentation skills', 'patient care',
}

SOFT_SKILLS_SET = SOFT_SKILLS_MASTER   # alias untuk kompatibilitas BQ1
SOFT_SKILLS_DB  = SOFT_SKILLS_MASTER   # alias untuk feature engineering

print("✅ Imports selesai")


✅ Imports selesai


## 3. Data Wrangling
Membaca `Resume.csv` dan `job_postings.csv`, mengekstrak skills / experience /
education via regex, lalu menghasilkan dua dataset terstruktur.


In [ ]:
# ─── Regex patterns & lookup tables ─────────────────────────────────────────

SKILL_KEYWORDS = [
    # Programming & Tech Stack
    "python","sql","r","java","javascript","typescript","golang","scala","kotlin",
    "ruby","php","matlab","bash","shell","perl","swiftui","ios development",
    "react","angular","vue",r"node.js","django","flask","spring boot",r"c++",r"c#",
    "html","css","flutter","react native","android development",r"objective-c",

    # ML / AI / Data
    "machine learning","deep learning","nlp","natural language processing",
    "computer vision","tensorflow","pytorch","keras","scikit-learn","xgboost",
    "lightgbm","random forest","artificial intelligence","data mining","statistics",
    "mathematics",r"a/b testing","predictive modeling","data warehouse",

    # Cloud / DevOps
    "aws","gcp","azure","docker","kubernetes","terraform","jenkins",r"ci/cd",
    "devops","airflow","spark","hadoop","kafka","databricks","linux","unix",

    # Databases / BI
    "mysql","postgresql","mongodb","redis","elasticsearch","oracle","sqlite",
    "tableau",r"power bi","looker","excel","powerpoint","google analytics",
    "pandas","numpy","matplotlib","seaborn","plotly",

    # Engineering (Civil/Mech/Elec/Aviation)
    "autocad","solidworks","ansys","catia","revit","engineering design",
    "structural analysis","quality control","manufacturing","troubleshooting",
    "safety","gis","scada","plc programming","mechanical design","thermodynamics",
    "fluid mechanics","mechatronics","robotics","civil engineering","hvac",
    "aerospace engineering","aviation","construction management","automotive design",

    # Business / Mgmt / Ops
    "agile methodology","scrum","kanban","project management","stakeholder management",
    "crm","salesforce","erp","sap","six sigma","lean manufacturing","budgeting",
    "operations","business intelligence","strategy","strategic planning","pmp",

    # Finance, Accounting & Banking
    "financial modeling","accounting","gaap","vba","bloomberg","risk management",
    "auditing","taxation","financial analysis","payroll","accounts payable",
    "accounts receivable","ifrs","financial reporting","investment banking",
    "portfolio management","wealth management","equities","hedge funds","corporate finance",

    # Legal & Compliance
    "contract drafting","litigation","compliance","corporate law",
    "intellectual property","legal research","paralegal skills","advocacy",

    # Marketing, PR & Sales
    "seo","hubspot","social media","email marketing","copywriting","brand management",
    "b2b sales","b2c","lead generation","cold calling","google ads","facebook ads",
    "sem","ppc","content marketing","sales strategy","account management",
    "public relations","media relations","corporate communications",

    # Design, Arts & Creative
    "figma","adobe xd","ux","user research","wireframing","photoshop","illustrator",
    "typography","color theory","video editing","premiere pro","after effects",
    "animation","3d modeling","blender","maya","graphic design","art direction",

    # HR, Supply Chain & Logistics (inc. Agriculture)
    "recruitment","hris","supply chain","logistics","data entry","workday",
    "talent acquisition","onboarding","inventory management","employee relations",
    "performance management","procurement","vendor management","agriculture",
    "agronomy","farming","supply chain optimization",

    # Customer Service, Hospitality & Culinary
    "zendesk","ticketing system","dispute resolution","client retention","technical support",
    "customer service","hospitality","culinary skills","food safety",
    "restaurant management","event planning","guest relations",

    # Healthcare, Medical & Fitness
    "healthcare","patient care","surgery","clinical diagnostics","cpr","emr",
    "nursing","medical terminology","treatment planning","pharmacology",
    "rehabilitation","clinical research","hipaa","bls","acls","triage",
    "medical billing","patient assessment","anatomy","physiology","fitness training",

    # Education & Academic
    "education","teaching","research","public speaking","curriculum development",
    "mentoring","instructional design","tutoring","lesson planning",
    "classroom management","student evaluation","special education","elearning",

    # Universal Soft Skills (Specific / Measurable)
    "communication","leadership","analytical","problem solving","time management",
    "negotiation","teamwork","adaptability","critical thinking",
    "attention to detail","multitasking","conflict resolution","presentation skills",
]

SKILL_PATTERN = re.compile(r"\b(" + "|".join(SKILL_KEYWORDS) + r")\b", re.IGNORECASE)
EXP_PATTERN   = re.compile(
    r"(\d+)\+?\s*(?:to\s*\d+)?\s*(?:\-\s*\d+)?\s*years?\s*(?:of\s*)?"
    r"(?:experience|exp|work\s*experience|professional\s*experience)",
    re.IGNORECASE,
)
EDU_PATTERN = re.compile(
    r"\b(ph\.?d|doctorate|master\'?s?|m\.?s\.?c?|m\.?b\.?a?|m\.?eng?|"
    r"bachelor\'?s?|b\.?s\.?c?|b\.?a\.?|b\.?eng?|associate\'?s?|"
    r"high\s*school\s*diploma|hs\s*diploma|diploma)\b",
    re.IGNORECASE,
)

EDU_RANK_MAP = {
    "phd":5,"doctorate":5,"master's":4,"masters":4,"msc":4,"ms":4,"mba":4,"meng":4,
    "bachelor's":3,"bachelors":3,"bsc":3,"ba":3,"beng":3,
    "associate's":2,"associates":2,
    "high school diploma":1,"hs diploma":1,"diploma":1,
}
EXP_LEVEL_MAP = {
    "Internship":0,"Entry level":1,"Associate":2,
    "Mid-Senior level":5,"Director":8,"Executive":12,
}

SECTOR_RULES = [
    (r"\b(software|developer|devops|cloud|data\s*scientist|machine\s*learning|"
     r"ml\s*engineer|data\s*engineer|data\s*analyst|cybersecurity|network|"
     r"full\s*stack|backend|frontend|hadoop|blockchain|testing|sysadmin|"
     r"programmer|it\b|information\s*technology|web|database|system\s*admin)\b",
     "Technology & IT"),
    (r"\b(accountant|accounting|controller|cpa|auditor|financial\s*analyst|"
     r"tax\b|bookkeeper|payroll|billing|ledger|finance|accounts\s*payable|accounts\s*receivable)\b",
     "Finance & Accounting"),
    (r"\b(sales|account\s*executive|business\s*development|retail|inside\s*sales|"
     r"cashier|clerk|store|merchandiser|buyer|telemarketing|account\s*manager|wholesale)\b",
     "Sales & Retail"),
    (r"\b(marketing|seo|content|brand|digital\s*marketing|social\s*media|campaign|"
     r"public\s*relations|media|copywriter|communications?|event|pr\b|advertising)\b",
     "Marketing & PR"),
    (r"\b(hr\b|human\s*resources|recruiter|talent|people\s*ops|hris|onboarding|"
     r"staffing|benefits|compensation|employee\s*relations)\b",
     "Human Resources"),
    (r"\b(project|program|scrum|product\s*manager|product\s*owner|agile|pmo|operations|"
     r"product\s*management|process\s*manager|monitoring|evaluation)\b",
     "Project & Product Management"),
    (r"\b(nurse|physician|doctor|medical|clinical|healthcare|patient|surgeon|"
     r"therapist|pharmacist|dental|hearing|cardiac|audiolog|care\s*giver|counselor|"
     r"psychologist|optometrist|veterinar|phlebotomist|health|allied|crna|cna|cma)\b",
     "Healthcare & Medical"),
    (r"\b(lawyer|attorney|legal|paralegal|compliance|counsel|litigation|advocate|"
     r"law\b|contract\s*manager)\b",
     "Legal & Compliance"),
    (r"\b(teacher|professor|instructor|tutor|curriculum|academic|education|lecturer|"
     r"faculty|coach|trainer|school|principal)\b",
     "Education & Training"),
    (r"\b(mechanical|civil|electrical|structural|aerospace|chemical|industrial|"
     r"manufacturing|architect|construction\s*eng|estimating|estimator|draftsman|engineer\b|engineering)\b",
     "Engineering"),
    (r"\b(supply\s*chain|logistics|warehouse|procurement|inventory|shipping|receiving|"
     r"material|distribution|agriculture|agronomy|farming|forklift|loader|supply|purchasing|freight)\b",
     "Operations & Supply Chain"),
    (r"\b(designer|ux|ui|graphic|creative|art\b|animator|illustrator|photographer|video|"
     r"editor|writer|journalist|producer|multimedia|content\s*creator)\b",
     "Design, Media & Creative"),
    (r"\b(customer|support|call\s*center|client\s*success|help\s*desk|representative|"
     r"service\s*desk|front\s*desk|guest\s*service)\b",
     "Customer Service"),
    (r"\b(investment|banking|insurance|portfolio|wealth|equity|hedge|underwriter|"
     r"actuary|loan|mortgage|teller|claims|adjuster)\b",
     "Banking & Insurance"),
    (r"\b(chef|cook|bartender|server|waiter|waitress|barista|hostess|hospitality|"
     r"restaurant|food|culinary|hotel|banquet|catering)\b",
     "Food & Hospitality"),
    (r"\b(mechanic|technician|maintenance|janitor|custodian|housekeeping|cleaner|maid|"
     r"facilities|welder|carpenter|plumber|electrician|machinist|painter|mason|roofer|construction|hvac)\b",
     "Construction, Facilities & Trades"),
    (r"\b(pilot|aviation|flight|aerospace|driver|courier|delivery|transport|truck|transit|chauffeur)\b",
     "Transportation & Aviation"),
    (r"\b(security|officer|guard|police|firefighter|safety|patrol|investigator|military)\b",
     "Security & Public Safety"),
    (r"\b(real\s*estate|property|leasing|realtor|broker)\b",
     "Real Estate & Property"),
    (r"\b(scientist|researcher|chemist|biologist|lab\s*tech|laboratory|biology|chemistry|physics|research\s*assistant)\b",
     "Science & Research"),
    (r"\b(admin|administrative|receptionist|office\s*manager|data\s*entry|secretary|executive\s*assistant|clerical|office\s*assistant)\b",
     "Admin & Clerical"),
    (r"\b(consultant|advisor|strateg|business\s*analyst|management\s*consulting)\b",
     "Business Strategy & Consulting"),
]

RESUME_SECTOR_MAP = {
    "INFORMATION-TECHNOLOGY": "Technology & IT",
    "BUSINESS-DEVELOPMENT":   "Sales & Retail",
    "FINANCE":                "Finance & Accounting",
    "ACCOUNTANT":             "Finance & Accounting",
    "ENGINEERING":            "Engineering",
    "SALES":                  "Sales & Retail",
    "MARKETING":              "Marketing & PR",
    "HR":                     "Human Resources",
    "HEALTHCARE":             "Healthcare & Medical",
    "ADVOCATE":               "Legal & Compliance",
    "FITNESS":                "Healthcare & Medical",
    "AVIATION":               "Transportation & Aviation",
    "CHEF":                   "Food & Hospitality",
    "ARTS":                   "Design, Media & Creative",
    "TEACHER":                "Education & Training",
    "BANKING":                "Banking & Insurance",
    "CONSULTANT":             "Business Strategy & Consulting",
    "DIGITAL-MEDIA":          "Marketing & PR",
    "AUTOMOBILE":             "Engineering",
    "AGRICULTURE":            "Operations & Supply Chain",
    "CONSTRUCTION":           "Engineering",
    "PUBLIC-RELATIONS":       "Marketing & PR",
    "BPO":                    "Customer Service",
    "APPAREL":                "Sales & Retail",
    "DESIGNER":               "Design, Media & Creative",
    "REACT-DEVELOPER":        "Technology & IT",
    "SQL-DEVELOPER":          "Technology & IT",
    "DOTNET-DEVELOPER":       "Technology & IT",
    "ETL-DEVELOPER":          "Technology & IT",
    "JAVA-DEVELOPER":         "Technology & IT",
    "PYTHON-DEVELOPER":       "Technology & IT",
    "HADOOP":                 "Technology & IT",
    "NETWORK-SECURITY-ENGINEER": "Technology & IT",
    "OPERATIONS-MANAGER":     "Project & Product Management",
    "TESTING":                "Technology & IT",
    "SAP-DEVELOPER":          "Technology & IT",
    "BLOCKCHAIN":             "Technology & IT",
    "PMO":                    "Project & Product Management",
    "DATA-SCIENCE":           "Technology & IT",
    "MECHANICAL-ENGINEER":    "Engineering",
    "ELECTRICAL-ENGINEERING": "Engineering",
    "CIVIL-ENGINEER":         "Engineering",
    "HEALTH-AND-FITNESS":     "Healthcare & Medical",
}

# ─── Helper functions ─────────────────────────────────────────────────────────

def normalize_edu(raw):
    if not raw: return "Bachelor's"
    r = raw.lower().replace("'", "").replace(".", "").replace(" ", "")
    if "phd" in r or "doct" in r: return "PhD"
    if "master" in r or r in ("msc", "ms", "mba", "meng"): return "Master's"
    if "bachelor" in r or r in ("bsc", "beng", "ba"):  # ← tambahkan 'ba'
        return "Bachelor's"
    if "associate" in r: return "Associate's"
    return "High School / Diploma"

def extract_skills(text):
    if not isinstance(text, str): return []
    return list(set(s.lower() for s in SKILL_PATTERN.findall(text)))


def extract_experience_years_resume(text):
    """Untuk resume: ambil total/max pengalaman yang disebut."""
    if not isinstance(text, str): return 0
    m = EXP_PATTERN.findall(text)
    if m: return int(max(int(x) for x in m))  # ← max untuk resume
    simple = re.findall(r"(?:minimum\s+of\s+)?(\d{1,2})\s+years?", text, re.IGNORECASE)
    valid = [int(x) for x in simple if 0 < int(x) <= 30]
    return int(max(valid)) if valid else 0     # ← max untuk resume

def extract_experience_years_job(text):
    """Untuk job posting: ambil minimum syarat pengalaman."""
    if not isinstance(text, str): return 0
    m = EXP_PATTERN.findall(text)
    if m: return int(min(int(x) for x in m))  # ← min tetap benar untuk JD
    simple = re.findall(r"(?:minimum\s+of\s+)?(\d{1,2})\s+years?", text, re.IGNORECASE)
    valid = [int(x) for x in simple if 0 < int(x) <= 30]
    return int(min(valid)) if valid else 0

def extract_education_level(text):
    if not isinstance(text, str): return "Bachelor's"
    m = EDU_PATTERN.findall(text)
    if not m: return "Bachelor's"
    best = max(m, key=lambda x: EDU_RANK_MAP.get(
        x.lower().replace("'","").replace(".","").replace(" ",""), 0))
    return normalize_edu(best)

def infer_sector(title):
    if not isinstance(title, str): return "Other"
    for pattern, sector in SECTOR_RULES:
        if re.search(pattern, title, re.IGNORECASE): return sector
    return "Other"

def clean_job_title(title: str) -> str:
    if pd.isna(title) or not isinstance(title, str): return ""
    t = title.strip()

    t = re.sub(r'\b(3|2)[dD]\b', r'\1D_TEMP', t)
    t = re.sub(r'\bB2B\b', 'B2B_TEMP', t, flags=re.IGNORECASE)

    garbage_phrases = [
        r'hiring\s+event', r'job\s+fair', r'virtual\s+hiring', r'multiple\s+positions',
        r'multiple\s+open\s+positions', r'various\s+positions', r'general\s+application',
        r'open\s+positions?', r'positions?\s+available'
    ]
    for p in garbage_phrases:
        if re.search(p, t, re.IGNORECASE): return ""

    t = re.sub(r'\b(?:W2|1099|C2C|No C2C|FT|PT|PRN|Per Diem|Flex|Ojt)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:day|night|evening|weekend|morning|afternoon|swing|rotating|flexible|hourly)?\s*shifts?\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b\d+\s*(?:hr|hour|hrs|wk|week|day|month|year|yr)s?\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b\d+\s*[aApP][mM]\b', '', t)
    t = re.sub(r'\b\d+/\d+\b', '', t)
    t = re.sub(r'^\d+[a-zA-Z]?\s+', '', t)
    t = re.sub(r'^(?:urgent|hiring|looking for|we are hiring|join(?: our| my| the)?(?: [A-Za-z]+)* team(?: as)?|now hiring|vacancy(?: for)?|apply(?: now)?(?: for)?|get hired as|opportunity(?: for)?|wanted|urgently hiring)\b\s*[:\-]?\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\bwith\s+.*(?:certification|experience|exp|background|skills?|knowledge|benefits)\b.*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\bwith\s+[a-zA-Z0-9\s\-\&]+$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+O2C Experience\b', ' O2C', t, flags=re.IGNORECASE)
    t = re.sub(r'\s*[-|/:]\s*(?:apply(?: now)?|click here|join us|immediate start|sign on bonus|bonus|urgent).*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s*[\(\[\{][^\)\]\}]*[\)\]\}]', '', t)
    t = re.sub(r'\b(?:remote|work remotely|working remotely|hybrid|contract|freelance|part[- ]time|full[- ]time|temporary|temp|internship|intern|travel|traveler|traveling)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:job|position|opening|role|opportunity|posting|available|employment|assignment)\b.*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:at|in|with)\b\s+(?:remote|onsite|hybrid|office|team|company|hospital|clinic|restaurant|school|center|centre|store|site|location|branch|headquarters|hq|warehouse)\b.*$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:at|in|with)\b\s+[A-Za-z .]+,\s*[A-Z]{2}(?:\s*.*)?$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:at|in|with)\b\s+[A-Za-z .]+,\s*[A-Za-z .]+$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s*-\s*[A-Za-z .]+,\s*[A-Z]{2}(?:\s*.*)?$', '', t)
    t = re.sub(r'\s*,\s*[A-Za-z .]+,\s*[A-Z]{2}(?:\s*.*)?$', '', t)
    t = re.sub(r'\s*-\s*(?:make\b.*|\$\s*\d+.*|\d+.*(?:week|wk|hour|hr|month|year|k)\b.*)$', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:Acme|Brunswick|Cincinnati|San Antonio|Central Us|Coxsackie|Torrance|Dallas|Houston|Denver|Hingham|South Mountain|Mesquite|Eden Prairie|New York|Redmond)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:days|monday|through|friday|9am|need|only|ca only)\b', '', t, flags=re.IGNORECASE)

    if re.search(r'[-:]', t):
        parts = [p.strip() for p in re.split(r'[-:]', t) if p.strip()]
        if parts:
            first_lower = parts[0].lower()
            if first_lower in ['north america', 'na', 'apac', 'emea', 'latam', 'global', 'us', 'usa', 'uk'] or \
               'department' in first_lower or 'division' in first_lower or len(parts[0]) <= 3:
                t = parts[1] if len(parts) > 1 else parts[0]
            else:
                t = parts[0]
    t = re.sub(r'\s*,.*$', '', t)
    t = re.sub(r'\b(?:IC|LV|L|T|Tier|Grade|Step|Level|Class)\s*\d*\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?=.*\d)(?=.*[a-zA-Z])[a-zA-Z0-9]{4,}\b', '', t)
    t = re.sub(r'\b[A-Za-z]*\d{4,}[A-Za-z]*\b', '', t)
    t = re.sub(r'\b\d+(?:st|nd|rd|th)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b\d+\b', '', t)
    t = re.sub(r'\b(?:I|II|III|IV|V|VI|VII|VIII|IX|X)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:[UO]65|Under 65|Over 65)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:Br|Dept|Div|Multi[- ]?(?:Site|Unit|Location|Property)?)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(?:Oracle|EBS|SAP|Salesforce|Microsoft|MS|AWS|Google|Workday|ServiceNow|Adobe|Cisco|IBM|PeopleSoft|NetSuite|Dynamics|Pizza Hut)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(Data|Order)\s+Entry\b', r'\1_Entry_Temp', t, flags=re.IGNORECASE)
    t = re.sub(r'\bEntry\b', '', t, flags=re.IGNORECASE)
    t = t.replace('_Entry_Temp', ' Entry')
    t = re.sub(r'\s*(?:&|and)\s+[A-Za-z]+\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\bsdet\b', 'Software Development Engineer in Test', t, flags=re.IGNORECASE)
    t = re.sub(r'\bdba\b', 'Database Administrator', t, flags=re.IGNORECASE)
    t = re.sub(r'\bbsa\b', 'Business Systems Analyst', t, flags=re.IGNORECASE)
    t = re.sub(r'\bqa\b', 'Quality Assurance', t, flags=re.IGNORECASE)
    t = re.sub(r'\bpm\b', 'Project Manager', t, flags=re.IGNORECASE)
    t = re.sub(r'\bmgr\b', 'Manager', t, flags=re.IGNORECASE)
    t = re.sub(r'\bspec\b', 'Specialist', t, flags=re.IGNORECASE)
    t = re.sub(r'\bbim\b', 'Building Information Modeling Specialist', t, flags=re.IGNORECASE)
    t = re.sub(r'\bbmet\b', 'Biomedical Equipment Technician', t, flags=re.IGNORECASE)
    t = re.sub(r'\bct\s+rad\b', 'CT Radiologic Technologist', t, flags=re.IGNORECASE)
    t = re.sub(r'\bekg\b', 'Electrocardiogram', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcrna\b', 'Certified Registered Nurse Anesthetist', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcma\b', 'Certified Medical Assistant', t, flags=re.IGNORECASE)
    t = re.sub(r'\bvp\b', 'Vice President', t, flags=re.IGNORECASE)
    t = re.sub(r'\bsvp\b', 'Senior Vice President', t, flags=re.IGNORECASE)
    t = re.sub(r'\bavp\b', 'Assistant Vice President', t, flags=re.IGNORECASE)
    t = re.sub(r'\bceo\b', 'Chief Executive Officer', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcto\b', 'Chief Technology Officer', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcfo\b', 'Chief Financial Officer', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcoo\b', 'Chief Operating Officer', t, flags=re.IGNORECASE)
    t = re.sub(r'\brn\b', 'Registered Nurse', t, flags=re.IGNORECASE)
    t = re.sub(r'\ber\b', 'Emergency Room', t, flags=re.IGNORECASE)
    t = re.sub(r'\bicu\b', 'Intensive Care Unit', t, flags=re.IGNORECASE)
    t = re.sub(r'\blpn\b', 'Licensed Practical Nurse', t, flags=re.IGNORECASE)
    t = re.sub(r'\bcna\b', 'Certified Nursing Assistant', t, flags=re.IGNORECASE)
    t = re.sub(r'\bnp\b', 'Nurse Practitioner', t, flags=re.IGNORECASE)
    t = re.sub(r'\bmed\s*surg\b', 'Medical Surgical', t, flags=re.IGNORECASE)
    t = re.sub(r'\b(Senior|Sr|Junior|Jr|Lead|Principal|Chief|Head|Staff|Expert|Master|Trainee|Apprentice|Experienced|Certified|Corrections)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+\d+[a-zA-Z]{1,2}$', '', t)
    t = t.replace('3D_TEMP', '3D').replace('2D_TEMP', '2D').replace('B2B_TEMP', 'B2B')
    t = re.sub(r'^[\s\-,;:]+|[\s\-,;:]+$', '', t)
    t = re.sub(r'^(?:a|an|the)\s+', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip().title()

    t_lower = t.lower()
    if 'data scien' in t_lower: return 'Data Scientist'
    if 'data analy' in t_lower: return 'Data Analyst'
    if 'data eng' in t_lower: return 'Data Engineer'
    if 'machine learning' in t_lower or 'ai eng' in t_lower: return 'Machine Learning Engineer'
    if 'full stack' in t_lower: return 'Full Stack Developer'
    if 'front end' in t_lower or 'frontend' in t_lower: return 'Frontend Developer'
    if 'back end' in t_lower or 'backend' in t_lower: return 'Backend Developer'
    if 'software' in t_lower and ('eng' in t_lower or 'dev' in t_lower): return 'Software Engineer'
    if 'account man' in t_lower: return 'Account Manager'
    if 'project man' in t_lower: return 'Project Manager'
    if 'product man' in t_lower: return 'Product Manager'
    if 'business analy' in t_lower: return 'Business Analyst'
    if 'registered nurse' in t_lower: return 'Registered Nurse'
    if 'customer serv' in t_lower or 'customer supp' in t_lower: return 'Customer Service Representative'
    if 'summer associate' in t_lower: return 'Summer Associate'
    if '&e coor' in t_lower or 'm&e coor' in t_lower: return 'Monitoring And Evaluation Coordinator'

    while True:
        prev_t = t
        t = re.sub(r'\b(?:with|or|and|for|to|from|of|non|an|a|the|in|at)\b$', '', t, flags=re.IGNORECASE).strip()
        if t == prev_t: break

    invalid_words = {
        'in', 'at', 'with', 'join', 'apply', 'now', 'hiring', 'role', 'job', 'position',
        'north america', 'apac', 'emea', 'latam', 'global', 'usa', 'us', 'uk',
        'br', 'dept', 'div', 'multi', 'entry', 'advanced', 'back', 'class', 'client', 'club',
        'control', 'emergency', 'epic', 'ewm', 'fico', 'field based', 'extra help', 'day shift',
        'days', 'chair', 'dem', 'need', 'only', 'ca only', 'day on the quad', '1l summer associate',
        'years need', 'corrections', 'to', 'from', 'am', 'pm', '11a medical', 'bio', 'chapter', 'level', 'tier', 'grade', ''
    }
    if t.lower() in invalid_words: return ""
    if re.search(r'^\d+[A-Za-z]\s+Medical$', t, flags=re.IGNORECASE): return ""
    vague_single_words = {
        'new', 'old', 'best', 'wanted', 'urgent', 'required', 'immediate', 'start',
        'part', 'full', 'main', 'need', 'to', 'from', 'am', 'pm', 'and', 'for', 'non',
        'day', 'mid', 'bun', 'dir', 'eng', 'rep', 'tow', 'inc', 'llc', 'corp', 'big', 'small', 'all', 'any', 'level', 'tier', 'grade'
    }
    if len(t.split()) == 1 and (t.lower() in vague_single_words or (len(t) <= 2 and t.lower() not in ['it', 'hr', 'rn', 'pr', 'vp', 'er'])): return ""
    if len(t.split()) > 5: return ""
    if not re.match(r'^[a-zA-Z0-9\s\-\&]+$', t): return ""
    return t

def assess_data_quality(df, name="Dataset"):
    print(f"\n{'='*60}")
    print(f"  DATA QUALITY REPORT: {name}")
    print(f"{'='*60}")
    print(f"  Shape      : {df.shape}")
    print(f"  Duplicates : {df.duplicated().sum():,}")
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    qdf     = pd.DataFrame({"Missing": missing, "Missing%": pct, "dtype": df.dtypes})
    has_m   = qdf[qdf["Missing"] > 0].sort_values("Missing%", ascending=False)
    print(has_m.to_string() if len(has_m) else "  No missing values ✅")

print("✅ Wrangling helpers defined")


✅ Wrangling helpers defined


In [ ]:
# Load raw data

raw_resume = pd.read_csv(RESUME_PATH)
raw_jobs   = pd.read_csv(JOBS_PATH)
print(f"✅ Resume.csv       loaded : {raw_resume.shape[0]:,} rows | cols: {raw_resume.columns.tolist()}")
print(f"✅ job_postings.csv loaded : {raw_jobs.shape[0]:,} rows  | {len(raw_jobs.columns)} columns")

assess_data_quality(raw_resume, "Resume.csv")
assess_data_quality(raw_jobs,   "job_postings.csv")


✅ Resume.csv       loaded : 2,484 rows | cols: ['ID', 'Resume_str', 'Resume_html', 'Category']
✅ job_postings.csv loaded : 33,246 rows  | 28 columns

  DATA QUALITY REPORT: Resume.csv
  Shape      : (2484, 4)
  Duplicates : 0
  No missing values ✅

  DATA QUALITY REPORT: job_postings.csv
  Shape      : (33246, 28)
  Duplicates : 0
                            Missing  Missing%    dtype
skills_desc                   32909     98.99   object
closed_time                   32074     96.47  float64
med_salary                    31005     93.26  float64
remote_allowed                28444     85.56  float64
min_salary                    22135     66.58  float64
max_salary                    22135     66.58  float64
pay_period                    19894     59.84   object
currency                      19894     59.84   object
compensation_type             19894     59.84   object
applies                       17008     51.16  float64
posting_domain                13558     40.78   object
applica

In [ ]:
# Clean Resume - candidate_df

def clean_resume_data(df):
    df = df.copy()
    before = len(df)
    df.drop_duplicates(subset=["ID"], inplace=True)
    df.dropna(subset=["Resume_str"], inplace=True)
    df["candidate_skills"] = df["Resume_str"].apply(extract_skills)
    df["experience_years"] = df["Resume_str"].apply(extract_experience_years_resume)
    df["education_level"]  = df["Resume_str"].apply(extract_education_level)
    df["skill_count"]      = df["candidate_skills"].apply(len)
    df.rename(columns={"Category": "industry_sector", "ID": "candidate_id"}, inplace=True)
    df["industry_sector"]  = df["industry_sector"].str.upper().map(
        lambda x: RESUME_SECTOR_MAP.get(x, x.replace("-", " ").title()))
    print(f"✅ Resume cleaned : {len(df):,} kandidat (removed {before-len(df)} duplicates) "
          f"| avg skills/resume: {df['skill_count'].mean():.1f}")
    return df[["candidate_id","industry_sector","candidate_skills",
               "experience_years","education_level","skill_count"]]

candidate_df = clean_resume_data(raw_resume)
candidate_df.head(3)


✅ Resume cleaned : 2,484 kandidat (removed 0 duplicates) | avg skills/resume: 10.4


,candidate_id,industry_sector,candidate_skills,experience_years,education_level,skill_count
0,16852973,Human Resources,"[hospitality, operations, safety, medical term...",15,Associate's,18
1,22323967,Human Resources,"[operations, project management, public relati...",0,Master's,8
2,33176873,Human Resources,"[quality control, excel, hris, nursing, projec...",20,Master's,15


In [ ]:
# Build Industry Data - industry_df

def build_industry_data(df):
    df = df.copy()
    df.dropna(subset=["description"], inplace=True)
    df.drop_duplicates(subset=["job_id"], inplace=True)

    print("  Cleaning job titles ...")
    df["title"] = df["title"].apply(clean_job_title)
    df = df[df["title"] != ""]

    df["text_combined"] = df["description"].fillna("") + " " + df["skills_desc"].fillna("")

    print("  Extracting skills …")
    df["required_skills"] = df["text_combined"].apply(extract_skills)

    print("  Rule-based imputation untuk skill yang sparse …")
    def enrich(row):
        skills = row["required_skills"] if isinstance(row["required_skills"], list) else []
        title  = str(row["title"]).lower()
        if len(skills) < 2:
            if 'bank' in title or 'investment' in title or 'portfolio' in title or 'wealth' in title:
                skills.extend(['banking', 'financial analysis', 'risk management', 'communication'])
            elif 'chef' in title or 'restaurant' in title or 'food' in title or 'hospitality' in title:
                skills.extend(['customer service', 'food safety', 'teamwork', 'time management'])
            elif 'agriculture' in title or 'farm' in title or 'agronomy' in title:
                skills.extend(['agriculture', 'operations', 'problem solving', 'supply chain'])
            elif 'aviation' in title or 'aerospace' in title or 'pilot' in title:
                skills.extend(['aviation', 'safety', 'engineering', 'attention to detail'])
            elif 'public relations' in title or 'pr ' in title or 'media' in title:
                skills.extend(['public relations', 'communication', 'media relations', 'copywriting'])
            elif 'product' in title or 'project' in title or 'scrum' in title or 'pmo' in title:
                skills.extend(['project management', 'agile methodology', 'leadership', 'communication', 'strategy'])
            elif 'sales' in title or 'retail' in title or 'account executive' in title or 'buyer' in title:
                skills.extend(['communication', 'negotiation', 'crm', 'b2b sales', 'lead generation'])
            elif 'marketing' in title or 'seo' in title or 'brand' in title or 'digital' in title:
                skills.extend(['seo', 'content marketing', 'social media', 'communication', 'data analysis'])
            elif 'finance' in title or 'accountant' in title or 'auditor' in title or 'tax' in title:
                skills.extend(['accounting', 'financial reporting', 'excel', 'analytical', 'attention to detail'])
            elif 'hr ' in title or 'recruiter' in title or 'human resources' in title:
                skills.extend(['talent acquisition', 'employee relations', 'communication', 'onboarding'])
            elif 'legal' in title or 'attorney' in title or 'lawyer' in title or 'counsel' in title:
                skills.extend(['legal research', 'compliance', 'contract drafting', 'analytical', 'communication'])
            elif 'design' in title or 'creative' in title or 'art' in title or 'animator' in title:
                skills.extend(['graphic design', 'adobe xd', 'typography', 'attention to detail', 'creativity'])
            elif 'supply chain' in title or 'logistics' in title or 'warehouse' in title:
                skills.extend(['inventory management', 'procurement', 'operations', 'problem solving'])
            elif 'support' in title or 'service' in title or 'help desk' in title or 'representative' in title:
                skills.extend(['customer service', 'communication', 'problem solving', 'ticketing system'])
            elif 'surgeon' in title or 'doctor' in title or 'physician' in title or 'dentist' in title:
                skills.extend(['healthcare', 'patient care', 'clinical diagnostics', 'treatment planning', 'communication'])
            elif 'nurse' in title or 'medical' in title or 'care' in title or 'therapist' in title or 'fitness' in title:
                skills.extend(['patient care', 'cpr', 'bls', 'patient assessment', 'emr'])
            elif 'professor' in title or 'lecturer' in title or 'teacher' in title or 'instructor' in title:
                skills.extend(['teaching', 'lesson planning', 'curriculum development', 'public speaking', 'communication'])
            elif 'civil' in title or 'structural' in title or 'architect' in title or 'construction' in title:
                skills.extend(['autocad', 'structural analysis', 'engineering design', 'project management'])
            elif 'technician' in title or 'engineer' in title or 'maintenance' in title or 'mechanic' in title:
                skills.extend(['troubleshooting', 'engineering design', 'safety', 'problem solving'])
            elif 'data' in title or 'analyst' in title or 'scientist' in title or 'developer' in title or 'software' in title:
                skills.extend(['sql', 'data analysis', 'problem solving', 'analytical', 'agile methodology'])
            else:
                skills.extend(['communication', 'teamwork', 'adaptability', 'problem solving'])
        return list(set(skills))

    df["required_skills"] = df.apply(enrich, axis=1)

    print("  Extracting experience requirements …")
    df["exp_from_text"] = df["text_combined"].apply(extract_experience_years_job)
    df["exp_from_level"] = df["formatted_experience_level"].map(EXP_LEVEL_MAP).fillna(0).astype(int)
    df["minimum_experience_years"] = df.apply(
        lambda r: r["exp_from_text"] if r["exp_from_text"] > 0 else r["exp_from_level"], axis=1)

    print("  Extracting education levels …")
    df["education_level"]  = df["text_combined"].apply(extract_education_level)
    df["industry_sector"]  = df["title"].apply(infer_sector)

    title_counts = df["title"].value_counts()
    max_count    = title_counts.max()
    df["market_demand_score"] = df["title"].map(
        lambda t: round(1 + 9 * (title_counts.get(t, 1) / max_count), 2))
    df["avg_salary"] = df[["min_salary","max_salary"]].mean(axis=1)

    print(f"✅ Industry data : {len(df):,} job postings")
    return df[["job_id","industry_sector","title","required_skills",
               "minimum_experience_years","education_level","market_demand_score",
               "location","formatted_work_type","formatted_experience_level",
               "avg_salary","remote_allowed"]].rename(columns={"title":"job_title"})

industry_df = build_industry_data(raw_jobs)
industry_df.head(3)


  Cleaning job titles ...
  Extracting skills …
  Rule-based imputation untuk skill yang sparse …
  Extracting experience requirements …
  Extracting education levels …
✅ Industry data : 30,206 job postings


,job_id,industry_sector,job_title,required_skills,minimum_experience_years,education_level,market_demand_score,location,formatted_work_type,formatted_experience_level,avg_salary,remote_allowed
0,3757940104,Healthcare & Medical,Hearing Care Provider,"[excel, c, patient care, troubleshooting, heal...",1,Bachelor's,1.01,"Little River, SC",Full-time,Entry level,NaN,NaN
2,3757938019,Other,Manager,"[quality control, excel, operations, safety, p...",10,Bachelor's,3.56,"Bessemer, AL",Full-time,NaN,NaN,NaN
3,3757938018,Food & Hospitality,Cook,"[hospitality, performance management, education]",3,Bachelor's,1.49,"Aliso Viejo, CA",Full-time,Entry level,NaN,NaN


In [ ]:
# Simpan ke session storage
candidate_df.to_csv(f"{OUTPUT_DIR}/cocokin_candidate_data.csv", index=False)
industry_df.to_csv(f"{OUTPUT_DIR}/cocokin_industry_data.csv",   index=False)
print(f"💾 cocokin_candidate_data.csv  -> {len(candidate_df):,} rows")
print(f"💾 cocokin_industry_data.csv   -> {len(industry_df):,} rows")


💾 cocokin_candidate_data.csv  -> 2,484 rows
💾 cocokin_industry_data.csv   -> 30,206 rows


## 4. Feature Engineering
Fitur yang dibangun:
- cand_soft_skills       list        Soft skills milik kandidat
- cand_tech_skills       list        Tech skills milik kandidat
- req_soft_skills        list        Soft skills yang diminta lowongan
- req_tech_skills        list        Tech skills yang diminta lowongan
- skill_match_ratio      float 0-1   Weighted sum (TF-IDF Lokal) kecocokan skill
- missing_skills         list        Skills in req but not in cand
- missing_skill_count    int         len(missing_skills)
- experience_gap_years   int >=0     max(0, req_exp - cand_exp)
- edu_gap                int 0-5     max(0, rank(req_edu) - rank(cand_edu))
- salary_percentile      float 0-1   Persentil gaji job relatif terhadap sektor
- overall_fit_score      float 0-1   Composite score (belum dinormalisasi)
- scaled_fit_score       float 0-1   Normalized score using MinMaxScaler


In [ ]:
# Feature Engineering Helpers

EDU_RANK_FE = {
    "phd":5,"doctorate":5,"master's":4,"masters":4,"msc":4,"ms":4,"mba":4,"meng":4,
    "bachelor's":3,"bachelors":3,"bsc":3,"ba":3,"beng":3,
    "associate's":2,"associates":2,
    "high school / diploma":1,"high school diploma":1,"hs diploma":1,"diploma":1,
}

def edu_rank_fe(edu):
    return EDU_RANK_FE.get(str(edu).lower().strip(), 3)

def safe_parse(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(str(val))
    except: return [s.strip() for s in str(val).split(",") if s.strip()]

def split_skills(skills_list):
    soft = [s for s in skills_list if str(s).lower().strip() in SOFT_SKILLS_DB]
    tech = [s for s in skills_list if str(s).lower().strip() not in SOFT_SKILLS_DB]
    return soft, tech

def generate_sector_skill_weights(ind_df):
    """Hitung bobot IDF per sektor — skill langka di sektor itu bernilai lebih tinggi."""
    sector_weights = {}
    for sector, group in ind_df.groupby("sector_key"):
        all_req = []
        for sl in group["required_skills"]:
            all_req.extend([str(s).lower().strip() for s in sl if s])
        counts     = Counter(all_req)
        total_jobs = len(group)
        sector_weights[sector] = {
            skill: round(math.log(1 + total_jobs / (cnt + 1)), 4)
            for skill, cnt in counts.items()
        }
    return sector_weights

def weighted_skill_match_ratio(c_skills, r_skills, sector_weights, sector):
    c = set(str(s).lower().strip() for s in c_skills)
    r = set(str(s).lower().strip() for s in r_skills)
    if not r: return 0.0
    weights   = sector_weights.get(sector, {})
    total_w   = sum(weights.get(req, 1.0) for req in r)
    matched_w = sum(weights.get(m, 1.0) for m in (c & r))
    return round(matched_w / total_w, 4) if total_w > 0 else 0.0

def missing_skills_list(c_skills, r_skills):
    c = set(str(s).lower().strip() for s in c_skills)
    r = set(str(s).lower().strip() for s in r_skills)
    return sorted(r - c)

def get_balanced_samples(group):
    s = group.sort_values("scaled_fit_score", ascending=False)
    n = len(s)
    if n <= 9: return s
    mid = n // 2
    return pd.concat([
        s.head(3),
        s.iloc[max(0,mid-1):min(n,mid+2)],
        s.tail(3)
    ]).drop_duplicates(subset=["candidate_id","job_id"])

print("✅ Feature engineering helpers defined")


✅ Feature engineering helpers defined


In [ ]:
# Run Feature Engineering

c = candidate_df.copy()
i = industry_df.copy()

c["candidate_skills"] = c["candidate_skills"].apply(safe_parse)
i["required_skills"]  = i["required_skills"].apply(safe_parse)

c["sector_key"] = c["industry_sector"].str.upper().str.strip()
i["sector_key"] = i["industry_sector"].str.upper().str.strip()

print("🧮 Menghitung bobot TF-IDF per sektor …")
sector_weights = generate_sector_skill_weights(i)

print("📋 Sampling 50 jobs per sektor …")
sampled_parts = []
for sector, grp in i.groupby("sector_key"):
    sampled_parts.append(grp.sample(min(50, len(grp)), random_state=42).copy())
i_sampled = pd.concat(sampled_parts, ignore_index=True)

merged = c.merge(i_sampled, on="sector_key", suffixes=("_cand","_job"))
print(f"   Merged shape: {merged.shape}")

print("🗂️ Memecah Tech Skills & Soft Skills …")
merged[["cand_soft_skills","cand_tech_skills"]] = merged["candidate_skills"].apply(
    lambda x: pd.Series(split_skills(x)))
merged[["req_soft_skills","req_tech_skills"]] = merged["required_skills"].apply(
    lambda x: pd.Series(split_skills(x)))

exp_col      = "experience_years_cand" if "experience_years_cand" in merged.columns else "experience_years"
edu_cand_col = "education_level_cand"  if "education_level_cand"  in merged.columns else "education_level_x"
edu_job_col  = "education_level_job"   if "education_level_job"   in merged.columns else "education_level_y"

print("⚡ Menghitung skill match (Sector-based TF-IDF) …")
merged["skill_match_ratio"] = merged.apply(
    lambda r: weighted_skill_match_ratio(
        r["candidate_skills"], r["required_skills"], sector_weights, r["sector_key"]), axis=1)
merged["missing_skills"]       = merged.apply(
    lambda r: missing_skills_list(r["candidate_skills"], r["required_skills"]), axis=1)
merged["missing_skill_count"]  = merged["missing_skills"].apply(len)
merged["experience_gap_years"] = (merged["minimum_experience_years"] - merged[exp_col]).clip(lower=0)
merged["edu_gap"]              = merged.apply(
    lambda r: max(0, edu_rank_fe(r[edu_job_col]) - edu_rank_fe(r[edu_cand_col])), axis=1)

merged["salary_percentile"] = merged.groupby("sector_key")["avg_salary"].rank(pct=True).fillna(0.5)
merged["demand_n"]          = merged["market_demand_score"] / 10.0

# Formula overall_fit_score
merged["overall_fit_score"] = (
    merged["skill_match_ratio"] * 0.60       # 60% skill match
    + merged["demand_n"] * 0.10              # 10% market demand
    - merged["experience_gap_years"] * 0.02  # penalti gap pengalaman
    - merged["edu_gap"] * 0.05               # penalti gap edukasi
    + merged["salary_percentile"] * 0.10     # 10% bonus gaji kompetitif
).clip(0, 1).round(4)

print("📈 Min-Max Scaling …")
scaler = MinMaxScaler()
merged["scaled_fit_score"] = scaler.fit_transform(merged[["overall_fit_score"]]).round(4)

eng = merged.copy()
print(f"\n✅ Feature matrix : {eng.shape[0]:,} pairs | {eng.shape[1]} features")
print(f"   avg skill_match_ratio = {eng['skill_match_ratio'].mean():.3f}")
print(f"   avg overall_fit_score = {eng['overall_fit_score'].mean():.3f}")
print(f"   avg experience_gap    = {eng['experience_gap_years'].mean():.1f} yrs")


In [ ]:
# Simpan 2 output dataset utama
print("💾 Menyimpan output …")

eng.to_csv(f"{OUTPUT_DIR}/cocokin_engineered.csv", index=False)
print(f" 1. cocokin_engineered.csv     → {len(eng):,} baris")

ai_train = eng.groupby("candidate_id", group_keys=False).apply(get_balanced_samples)
ai_train = ai_train.sample(frac=1, random_state=42).reset_index(drop=True)
ai_train.to_csv(f"{OUTPUT_DIR}/cocokin_training_ai.csv", index=False)
print(f" 2. cocokin_training_ai.csv    → {len(ai_train):,} baris")


## 5. EDA & Visualisasi
### Fase 1 — Exploratory Data Analysis
Chart deskriptif untuk memahami distribusi data mentah.

### Fase 2 — Business Questions
| # | Pertanyaan |
|---|-----------|
| BQ1 | Top 5 missing *tech skills* pada kandidat S1 di sektor Teknologi |
| BQ2 | Proporsi Soft vs Tech Skills: Finance & Accounting vs Technology |
| BQ3 | Realitas paradoks syarat pengalaman lowongan "Entry Level" |


In [ ]:
# EDA 1 - Distribusi Lowongan: Top 5 Sektor Industri

top_sectors = industry_df['industry_sector'].value_counts().nlargest(5)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_sectors.index[::-1], top_sectors.values[::-1], color=PALETTE[:5])
ax.set_title("Distribusi Lowongan: Top 5 Sektor Industri Teratas",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Jumlah Lowongan Pekerjaan")

for bar, val in zip(bars, top_sectors.values[::-1]):
    ax.text(bar.get_width() + top_sectors.max()*0.01,
            bar.get_y() + bar.get_height()/2, f"{val:,}", va="center", fontsize=10)

plt.tight_layout()
plt.show()


# insight EDA 1

- Poin Temuan Utama: Di luar kategori campuran ("Other"), sektor Sales & Retail (3.266 lowongan) dan Healthcare & Medical (2.732 lowongan) memimpin volume permintaan tenaga kerja.
- Insight Bisnis: Pasar kerja saat ini sangat didorong oleh industri yang bersentuhan langsung dengan konsumen (customer-facing) dan pelayanan publik. Hal ini menunjukkan bahwa terlepas dari masifnya digitalisasi, peran manusia di sektor perdagangan dan kesehatan tetap menjadi penyerap tenaga kerja terbesar.

In [ ]:
# EDA 2 - Proporsi Model Kerja (Remote vs On-Site)

if 'remote_allowed' in industry_df.columns:
    remote_series = industry_df['remote_allowed'].fillna(0).astype(float)
    remote_counts = remote_series.map(
        lambda x: 'Full Remote' if x == 1 else 'On-Site / Hybrid'
    ).value_counts()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(
        remote_counts.values,
        explode=(0.05, 0),
        labels=remote_counts.index,  # ← label dari data, bukan hardcoded
        colors=[PALETTE[1], PALETTE[0]],
        autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    ax.set_title("Proporsi Model Kerja (Remote vs On-Site)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Insight EDA 2

- Poin Temuan Utama: Mayoritas absolut lowongan kerja menuntut kehadiran fisik, dengan 85,5% mematok sistem On-Site / Hybrid, dan hanya 14,5% yang mengizinkan Full Remote.
- Insight Bisnis: Era kerja jarak jauh (remote work) besar-besaran yang terjadi saat pandemi telah berakhir. Perusahaan kini telah kembali ke model kerja konservatif. Bagi pencari kerja, fleksibilitas lokasi kini menjadi sebuah kemewahan (privilese) yang sangat langka di pasar.


In [ ]:
# EDA 3 - Korelasi Pengalaman Kerja vs Gaji (USD)

if 'avg_salary' in industry_df.columns:
    df_sc = industry_df.dropna(subset=['avg_salary','minimum_experience_years'])
    df_sc = df_sc[df_sc['minimum_experience_years'] <= 15].sample(
        n=min(1000, len(df_sc)), random_state=42)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.regplot(x='minimum_experience_years', y='avg_salary', data=df_sc,
                scatter_kws={'alpha':0.4, 'color': PALETTE[0]},
                line_kws={'color': PALETTE[4]}, ax=ax)
    ax.set_title("Korelasi Pengalaman Kerja vs Gaji Bulanan (USD)",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel("Pengalaman Minimum (Tahun)")
    ax.set_ylabel("Estimasi Gaji Tahunan (USD)")
    plt.tight_layout()
    plt.show()


# Insight EDA 3

- Poin Temuan Utama: Garis tren (warna merah) menunjukkan kemiringan positif yang sangat konsisten antara tahun pengalaman minimum dengan estimasi gaji tahunan.
- Insight Bisnis: Pengalaman adalah mata uang paling berharga di pasar kerja. Grafik ini membuktikan bahwa perusahaan tidak segan membayar harga premium (jauh di atas rata-rata pasar) untuk memvalidasi jam terbang seorang kandidat. Ini juga yang menjadi dasar mengapa Fresh Graduate sering kesulitan menembus filter awal rekrutmen.

In [ ]:
# EDA 4 - Distribusi Level Pengalaman per Sektor Utama

top_3 = industry_df['industry_sector'].value_counts().nlargest(3).index
df_cl = industry_df[
    industry_df['industry_sector'].isin(top_3) &
    industry_df['formatted_experience_level'].isin(['Entry level','Mid-Senior level','Director'])
]

fig, ax = plt.subplots(figsize=(12, 6))
sns.countplot(data=df_cl, x='industry_sector', hue='formatted_experience_level',
              palette="viridis", ax=ax)
ax.set_title("Distribusi Level Pengalaman per Sektor Utama",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Sektor Industri")
ax.set_ylabel("Jumlah Lowongan")
ax.legend(title="Level Pengalaman", loc='upper right')
plt.tight_layout()
plt.show()


# Insight EDA 4

- Poin Temuan Utama: Pada ketiga sektor utama yang dianalisis, level Mid-Senior dan Entry level mendominasi pembukaan lowongan, sementara posisi Director sangat sedikit.
- Insight Bisnis: Grafik ini memvalidasi bentuk alami "Piramida Hierarki Korporat". Tingginya volume di Entry Level dan Mid-Senior menunjukkan bahwa tingkat perputaran karyawan (turnover rate) paling tinggi terjadi di level pelaksana teknis dan manajerial menengah.

In [ ]:
# EDA 5 - Insights Gaji Sektoral (USD)

if 'avg_salary' in industry_df.columns:
    sal = industry_df[industry_df["avg_salary"].notna() & (industry_df["avg_salary"] > 100)]
    if len(sal) >= 10:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].hist(sal["avg_salary"].clip(upper=120_000), bins=30,
                     color=PALETTE[2], edgecolor="white")
        axes[0].set_xlabel("Gaji Tahunan (USD)")
        axes[0].set_ylabel("Frekuensi")
        axes[0].set_title("Distribusi Gaji Tahunan (Capped $120k)", fontweight="bold")

        top_sal = sal.groupby("industry_sector")["avg_salary"].median().sort_values(ascending=False).head(8)
        axes[1].barh(top_sal.index[::-1], top_sal.values[::-1], color=PALETTE[3])
        axes[1].set_xlabel("Median Gaji Tahunan (USD)")
        axes[1].set_title("Median Gaji Tertinggi per Sektor", fontweight="bold")

        fig.suptitle("Insights Gaji Sektoral (Dalam USD)",
                     fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()


# Insight EDA 5

- Poin Temuan Utama: Sektor berbasis STEM (Sains, Teknologi, Engineering, Matematika) merajai puncak kompensasi finansial. Sektor Science & Research dan Technology & IT menawarkan nilai tengah (median) gaji tertinggi.
- Insight Bisnis: Pasar memberikan valuasi (reward) yang sangat mahal untuk keahlian analitis, teknis, dan penelitian yang spesifik. Meskipun volume lowongannya (pada Gambar 1) kalah dari Sales atau Healthcare, kualitas kompensasi di sektor STEM terbukti jauh lebih superior.

In [ ]:
# BQ1 - Top 5 Missing Tech Skills: Kandidat S1, Sektor Teknologi

edu_cand_col = "education_level_cand" if "education_level_cand" in ai_train.columns else "education_level_x"

df_bq1 = ai_train[
    (ai_train[edu_cand_col].str.contains("Bachelor", case=False, na=False)) &
    (ai_train['sector_key'] == "TECHNOLOGY & IT")  # sesuai SECTOR_RULES
].copy()

all_missing = [
    skill.title()
    for m_list in df_bq1['missing_skills']
    for skill in m_list
    if skill.lower() not in SOFT_SKILLS_SET
]
counts = Counter(all_missing).most_common(5)

if counts:
    skills, freq = zip(*counts)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(list(skills)[::-1], list(freq)[::-1], color=PALETTE[4])
    ax.set_xlabel("Frekuensi Kemunculan sebagai Missing Skill")S
    ax.set_title("BQ1: Top 5 Missing Tech Skills\n(Pelamar S1 — Sektor Teknologi)",
                 fontsize=13, fontweight="bold")
    for bar, val in zip(bars, list(freq)[::-1]):
        ax.text(bar.get_width() + max(freq)*0.01,
                bar.get_y() + bar.get_height()/2, f"{val:,}", va="center", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("  ℹ️  Data tidak cukup untuk BQ1.")


In [ ]:
# BQ2 - Proporsi Soft vs Tech Skills: Finance vs Technology

df_bq2 = industry_df[
    industry_df['industry_sector'].isin(['Finance & Accounting', 'Technology & IT'])
].copy()

def calc_ratio(skills):
    soft = sum(1 for s in skills if s.lower() in SOFT_SKILLS_SET)
    return pd.Series({'Soft Skills': soft, 'Tech Skills': len(skills) - soft})

ratios  = df_bq2['required_skills'].apply(calc_ratio)
grouped = pd.concat([df_bq2, ratios], axis=1)            .groupby('industry_sector')[['Soft Skills','Tech Skills']].sum()
grouped_pct = grouped.div(grouped.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 6))
grouped_pct.plot(kind='bar', stacked=True,
                 color=[PALETTE[1], PALETTE[6]], ax=ax, width=0.5)
ax.set_title("BQ2: Proporsi Syarat Keahlian (Soft vs Tech Skills)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Sektor Industri")
ax.set_ylabel("Persentase Kebutuhan (%)")
plt.xticks(rotation=0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

for p in ax.patches:
    w, h = p.get_width(), p.get_height()
    if h > 0:
        ax.text(p.get_x() + w/2, p.get_y() + h/2, f"{h:.1f}%",
                ha='center', va='center', color='white', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# BQ3 - Realitas Syarat Pengalaman Lowongan "Entry Level"

df_bq3    = industry_df[
    (industry_df['formatted_experience_level'] == 'Entry level') &
    (industry_df['minimum_experience_years'] >= 0)
].copy()
top5_sec  = df_bq3.groupby('industry_sector')['minimum_experience_years'].mean().nlargest(5).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_bq3[df_bq3['industry_sector'].isin(top5_sec)],
            y='industry_sector', x='minimum_experience_years',
            order=top5_sec, palette="rocket", ax=ax)

global_mean = df_bq3['minimum_experience_years'].mean()
ax.axvline(global_mean, color=PALETTE[4], linestyle='--', linewidth=2,
           label=f'Rata-rata ({global_mean:.2f} Tahun)')

ax.set_title("BQ3: Realitas Syarat Pengalaman Lowongan 'Entry Level'",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Sektor Target")
ax.set_xlabel("Ekspektasi Minimum Pengalaman Kerja (Tahun)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


# Pertanyaan Bisnis 1

- Bagi pelamar S1 di sektor Teknologi, 5 keahlian teknis yang paling sering "hilang" (tidak ada di CV mereka namun diminta oleh perusahaan) adalah DevOps (40), Safety (39), Javascript (37), Compliance (32), dan Angular (32).
- Lulusan baru (S1) umumnya hanya berfokus pada bahasa pemrograman dasar. Mereka tertinggal dalam tren infrastruktur modern (DevOps) dan framework web spesifik (Javascript/Angular). Selain itu, tingginya kemunculan Safety dan Compliance membuktikan bahwa perguruan tinggi jarang mengajarkan standar keamanan dan kepatuhan level enterprise, padahal industri sangat membutuhkannya. Fitur Cocokin sangat tepat untuk merekomendasikan skill-skill spesifik ini kepada user.

# Pertanyaan Bisnis 2

- Baik sektor Finance & Accounting maupun Technology & IT sangat mendominasi pencarian kandidat berbasis Tech Skills (di atas 75%). Namun, sektor Keuangan menuntut porsi Soft Skills (24,5%) yang lebih besar dibandingkan sektor IT (17,1%).
- Hard skills/Tech skills adalah syarat mutlak (deal-breaker) untuk bisa masuk ke kedua industri ini. Namun, data menunjukkan profil risiko pekerjaan yang berbeda: Sektor Keuangan membutuhkan kemampuan komunikasi, negosiasi, atau presentasi (Soft Skills) yang lebih tinggi karena interaksi dengan klien atau regulasi. Sementara itu, sektor IT jauh lebih pragmatis dan heavy pada keahlian teknis murni.

# Pertanyaan Bisnis 3

- Label "Entry Level" di pasar kerja adalah sebuah paradoks. Rata-rata nasional menunjukkan bahwa posisi Entry Level justru menuntut 3,26 Tahun pengalaman kerja. (Catatan: Terlihat ada banyak titik outlier ekstrim di atas 20 hingga 160 tahun, yang menandakan kotornya data mentah dari perusahaan sebelum dibersihkan).
- Terjadi Inflasi Pengalaman (Experience Inflation) yang masif di pasar kerja. Perusahaan melabeli lowongan sebagai Entry Level (biasanya untuk menekan budget gaji), namun menuntut kandidat yang sudah "siap tempur" dengan pengalaman 3 tahun. Data ini memvalidasi penderitaan Fresh Graduate yang selalu kalah saing, dan mengukuhkan urgensi produk Cocokin untuk mem-(bypass) kekurangan pengalaman tersebut melalui pembuktian skill yang akurat.

# Kesimpulan Analisis Pertanyaan Bisnis

- Ketertinggalan Kurikulum Akademis terhadap Kebutuhan Praktis (Hasil BQ1)

  Data Missing Tech Skills pada lulusan S1 sektor Teknologi membuktikan adanya jarak (gap) yang lebar antara apa yang diajarkan di kampus dan apa yang dicari industri. Lulusan baru terlalu fokus pada bahasa pemrograman murni, namun melupakan kompetensi operasional level enterprise. Ketiadaan skill seperti DevOps, Compliance, dan standar Safety di CV mereka menunjukkan bahwa Fresh Graduate tidak siap menghadapi ekosistem kerja modern yang kompleks tanpa adanya proses adaptasi dan upskilling mandiri.

- Profil Risiko Sektoral Menentukan Kebutuhan Soft Skills (Hasil BQ2)

  Analisis proporsi skill membuktikan bahwa pendekatan melamar kerja tidak bisa disamaratakan. Walaupun Tech Skills (Hard Skills) mendominasi seluruh sektor sebagai syarat mutlak (di atas 75%), setiap sektor memiliki tuntutan Soft Skills yang spesifik. Sektor Finance & Accounting menuntut Soft Skills (24,5%) jauh lebih tinggi dibandingkan sektor Technology & IT (17,1%). Artinya, kandidat dengan skill teknis dewa sekalipun bisa ditolak di sektor keuangan jika CV mereka tidak menunjukkan kemampuan negosiasi, komunikasi, atau pemahaman regulasi.

- Mitos Posisi "Entry-Level" dan Inflasi Pengalaman (Hasil BQ3)

  Ini adalah temuan yang paling memberatkan para pencari kerja pemula. Istilah "Entry-Level" (Tingkat Pemula) di pasar kerja saat ini terbukti hanyalah sebuah label. Pada realitasnya, rata-rata perusahaan mematok standar 3,26 tahun pengalaman kerja untuk posisi tersebut. Terjadi "Inflasi Pengalaman", di mana perusahaan ingin menekan budget gaji dengan melabeli posisi sebagai Entry-Level, namun menuntut kandidat yang sudah matang dan "siap pakai".

- Benang Merah
  
  Ketiga fakta di atas menciptakan "Tembok Rintangan" yang sangat tinggi bagi Fresh Graduate. Mereka dituntut memiliki skill enterprise modern (BQ1), harus menyesuaikan porsi soft skill sesuai sektor (BQ2), dan tiba-tiba dituntut memiliki jam terbang 3 tahun untuk posisi pemula (BQ3. Kesimpulan ini mempertegas bahwa kandidat butuh alat bantu objektif—seperti platform Cocokin—untuk memetakan kekurangan spesifik mereka dan mem-bypass tembok pengalaman tersebut melalui penguasaan skill yang 100% relevan dengan industri.

## 6. A/B Testing — Statistical Validation
Framework pengujian statistik multi-tahap:

| Tahap | Uji | Tujuan |
|-------|-----|--------|
| Fase 1 | D'Agostino K² | Uji normalitas distribusi |
| Fase 1 | Levene's Test | Uji homogenitas variansi |
| Fase 2 | Welch's T-Test | Uji parametrik (satu arah) |
| Fase 2 | Mann-Whitney U | Fallback non-parametrik |
| Fase 2 | Cohen's d | Ukuran efek (magnitude) |

**Kelompok:**
- **Group A (Control)** — Kandidat senior: `experience_gap == 0`
- **Group B (Variant)** — Kandidat junior: `experience_gap > 0`

**Target Metric:** `skill_match_ratio` (TF-IDF Weighted Score)  
**Hipotesis:** H₁: mean(B) > mean(A) — apakah TF-IDF berhasil mengequaliser kandidat junior?


In [ ]:
# Cohen's d helper

def calculate_cohens_d(group1, group2):
    """Cohen's d = (mean1 - mean2) / pooled_std"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    if pooled_std == 0: return 0.0
    return (group1.mean() - group2.mean()) / pooled_std

print("✅ A/B testing helper defined")


In [ ]:
# Ekstrak kelompok dari production DB

alpha   = 0.05
group_a = ai_train[ai_train["experience_gap_years"] == 0]["skill_match_ratio"].dropna()
group_b = ai_train[ai_train["experience_gap_years"] >  0]["skill_match_ratio"].dropna()

n_a, n_b   = len(group_a), len(group_b)
mean_a, mean_b = group_a.mean(), group_b.mean()
lift       = (mean_b - mean_a) / mean_a * 100 if mean_a > 0 else 0

if n_a < 8 or n_b < 8:
    print("⚠️  Sampel terlalu kecil, menggunakan mock data untuk demonstrasi …")
    np.random.seed(42)
    n_mock = 250
    gaps   = np.random.choice([0,1,2,3], size=n_mock, p=[0.4,0.3,0.2,0.1])
    smr = np.where(gaps == 0,
        np.random.normal(0.50, 0.12, n_mock),
        np.random.normal(0.50, 0.10, n_mock))

    mock   = pd.DataFrame({"experience_gap_years": gaps, "skill_match_ratio": smr})
    group_a = mock[mock["experience_gap_years"] == 0]["skill_match_ratio"]
    group_b = mock[mock["experience_gap_years"] >  0]["skill_match_ratio"]

    n_a, n_b = len(group_a), len(group_b)
    mean_a, mean_b = group_a.mean(), group_b.mean()
    lift = (mean_b - mean_a) / mean_a * 100 if mean_a > 0 else 0

# Fase 1: Uji Asumsi

print("=" * 72)
print("  FASE 1: STATISTICAL ASSUMPTION TESTING")
print("=" * 72)

stat_na, p_na = stats.normaltest(group_a) if n_a >= 8 else (0, 0)
stat_nb, p_nb = stats.normaltest(group_b) if n_b >= 8 else (0, 0)
is_norm_a     = p_na >= alpha
is_norm_b     = p_nb >= alpha
normality_ok  = is_norm_a and is_norm_b

print(f"  Normality (D'Agostino K²):")
print(f"    Group A : p = {p_na:.4f} → {'NORMAL ✅' if is_norm_a else 'NON-NORMAL ⚠️'}")
print(f"    Group B : p = {p_nb:.4f} → {'NORMAL ✅' if is_norm_b else 'NON-NORMAL ⚠️'}")

stat_lev, p_lev = stats.levene(group_a, group_b)
equal_var = p_lev >= alpha
print(f"  Variance Homogeneity (Levene):")
print(f"    Statistic : {stat_lev:.4f}")
print(f"    p-value   : {p_lev:.4f} → {'EQUAL ✅' if equal_var else 'UNEQUAL ⚠️ (Welch adjustment applied)'}")

# Fase 2: Hipotesis & Effect Size

print()
print("=" * 72)
print("  FASE 2: HYPOTHESIS TESTING & EFFECT SIZE")
print("=" * 72)

t_stat, p_welch = stats.ttest_ind(group_b, group_a, equal_var=False, alternative='greater')
u_stat, p_mwu   = stats.mannwhitneyu(group_b, group_a, alternative='greater')
cohen_d         = calculate_cohens_d(group_b, group_a)

welch_sig = p_welch < alpha
mwu_sig   = p_mwu   < alpha

if   abs(cohen_d) < 0.2: eff_label = "Negligible"
elif abs(cohen_d) < 0.5: eff_label = "Small"
elif abs(cohen_d) < 0.8: eff_label = "Medium"
else:                    eff_label = "Large 🚀 (Extremely Strong Effect)"

chosen_test = "Welch's T-Test" if normality_ok else "Mann-Whitney U (Non-Parametric Fallback)"
final_p     = p_welch if normality_ok else p_mwu
definitive  = final_p < alpha

print(f"  Sample Distribution:")
print(f"    Group A (Senior)  : n={n_a:,} | mean={mean_a:.4f} | std={group_a.std():.4f}")
print(f"    Group B (Junior)  : n={n_b:,} | mean={mean_b:.4f} | std={group_b.std():.4f}")
print(f"    Observed Lift     : {lift:+.2f}%")
print(f"  Test Statistics:")
print(f"    Welch's T-test    : p = {p_welch:.6e}  [Significant: {str(welch_sig).upper()}]")
print(f"    Mann-Whitney U    : p = {p_mwu:.6e}  [Significant: {str(mwu_sig).upper()}]")
print(f"    Cohen's d         : {cohen_d:.4f} → {eff_label}")
print(f"  Framework Decision:")
print(f"    Primary Test      : {chosen_test}")
print(f"    Definitive p-value: {final_p:.6e}")
print(f"    Significant?      : {'YES ✅' if definitive else 'NO ❌'} (α={alpha})")
print("=" * 72)

print("\n  📊 BUSINESS INTERPRETATION:")
if definitive:
    print("    TOLAK H₀ — TF-IDF terbukti sebagai equalizer berbasis merit.")
    print("    Kandidat junior yang masuk Top 5 memiliki skill density lebih tinggi,")
    print("    mengimbangi defisit pengalaman mereka dengan efek yang signifikan.")
else:
    print("    GAGAL TOLAK H₀ — Pembobotan skill belum cukup mengkompensasi gap pengalaman.")
    print("    Perlu optimasi hyperparameter pada formula fit score.")
print("=" * 72)

print("⚠️  PERHATIAN: Menggunakan mock data NETRAL untuk demonstrasi framework.")
print("    Hasil statistik di bawah TIDAK mencerminkan data nyata.")
print("    Jalankan dengan data aktual untuk hasil yang valid.\n")

In [ ]:
# Visualisasi Hasil A/B Test

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel kiri: Probability Density Distribution
axes[0].hist(group_a, bins=25, alpha=0.6, color="#4C9BE8", edgecolor="white",
             label=f"Group A (Senior) μ={mean_a:.3f}", density=True)
axes[0].hist(group_b, bins=25, alpha=0.6, color="#F97316", edgecolor="white",
             label=f"Group B (Junior) μ={mean_b:.3f}", density=True)
axes[0].axvline(mean_a, color="#1D4ED8", linestyle="--", lw=2)
axes[0].axvline(mean_b, color="#B91C1C", linestyle="--", lw=2)
axes[0].set_xlabel("Skill Match Score (TF-IDF Value Range)")
axes[0].set_ylabel("Probability Density Profile")

annot = (f"Test: {chosen_test.split(' (')[0]}\n"
         f"p-val: {final_p:.4e}\n"
         f"Cohen's d: {cohen_d:.2f} ({eff_label.split(' ')[0]})")
axes[0].text(0.05, 0.92, annot, transform=axes[0].transAxes, fontsize=9.5,
             verticalalignment='top',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8FAFC',
                       alpha=0.9, edgecolor='#CBD5E1'))
axes[0].set_title("Probability Density Distribution Comparison",
                  fontweight="bold", fontsize=11)
axes[0].legend(loc="upper right")

# Panel kanan: Boxplot Quantile Analysis
bplot = axes[1].boxplot(
    [group_a, group_b],
    labels=["Group A\n(No Exp Gap)", "Group B\n(Has Exp Gap)"],
    patch_artist=True, medianprops=dict(color="black", lw=2.5))
for patch, color in zip(bplot['boxes'], ["#D1E9FF","#FFEDD5"]):
    patch.set_facecolor(color)
    patch.set_edgecolor("#64748B")
axes[1].set_ylabel("Quantile Score Range (TF-IDF Metric)")
axes[1].set_title(f"Quantile Boxplot Analysis (Lift: {lift:+.2f}%)",
                  fontweight="bold", fontsize=11)
axes[1].grid(axis='y', linestyle=':', alpha=0.6)

fig.suptitle("Cocokin A/B Testing Matrix — Algorithmic Fairness Audit",
             fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


# Insight A/B Testing (Evaluasi algoritma dan Realitas kesenjangan junior)

Dari hasil pengujian asumsi statistik (Mann-Whitney U Test) dan visualisasi distribusi (Probability Density & Boxplot), kita mendapatkan temuan yang sangat krusial mengenai perilaku algoritma TF-IDF terhadap kandidat Junior (Fresh Graduate) di pasar kerja sesungguhnya. Berikut adalah 4 insight utama dari pengujian ini:

1. Runtuhnya Asumsi "Keahlian Bisa Langsung Menutup Kurangnya Pengalaman"
  - Terdapat penurunan (Lift) skor kecocokan sebesar -24,32% pada kelompok Junior dibandingkan Senior. Rata-rata skor Junior hanya di angka 0.294, tertinggal jauh dari Senior yang berada di angka 0.389. Efek kesenjangan ini juga tervalidasi oleh nilai Cohen's d (-0.30) yang menunjukkan dampak negatif yang nyata, meskipun skalanya tergolong kecil-menengah.
  - Awalnya, algoritma pembobotan Sector-TF-IDF diharapkan mampu mengatrol skor kandidat Junior yang memiliki banyak skill agar bisa setara dengan Senior. Namun, data membuktikan sebaliknya. Pengalaman kerja secara alami dan organik membangun perbendaharaan skill (kata kunci) yang jauh lebih kaya di dalam CV seorang kandidat. Akibatnya, sistem yang murni objektif menilai skill (seperti algoritma Cocokin saat ini) akan tetap memenangkan kandidat Senior.
2. "Kegagalan" Hipotesis adalah Validasi Bisnis Terkuat
  - Uji statistik menghasilkan p-value = 1.000, yang berarti kita Gagal Menolak H₀ (Hipotesis Nol). Secara ilmiah, algoritma gagal membuktikan bahwa kandidat Junior memiliki kepadatan skill yang sama atau lebih baik dari Senior.
  - Dalam konteks riset produk, "kegagalan" ini justru merupakan penemuan yang cemerlang. Ini adalah bukti matematis transparan bahwa penderitaan Fresh Graduate yang kesulitan mencari kerja bukan sekadar mitos atau akibat HRD yang terlalu ketat menetapkan batas usia. Fresh Graduate memang terbukti secara kuantitatif "miskin" perbendaharaan skill industri.
3. Urgensi Pergeseran Produk (Dari "Pencari Kerja" menjadi "Mentor Karier")
  - Karena sistem telah membuktikan bahwa Junior tertinggal 24%, platform Cocokin tidak bisa hanya berfungsi sebagai tempat matchmaking (pencocokan) lowongan biasa. Jika hanya mencocokkan, Junior akan selalu kalah di ranking terbawah.
  - Penemuan ini memberikan landasan (business justification) yang absolut bahwa fitur Rekomendasi Missing Skills & Upskilling adalah nyawa utama dari aplikasi ini. Cocokin harus hadir untuk memberitahu pengguna: "Anda tertinggal 24% dari kandidat lain karena Anda tidak memiliki skill X, Y, dan Z. Pelajari ini untuk naik peringkat."
4. Evaluasi Teknis: Kebutuhan Optimasi Hyperparameter
  - Distribusi kedua kelompok terbukti tidak normal (p=0.0000) dan memiliki variansi yang timpang (Unequal Variance) karena sebaran skor Senior jauh lebih lebar dan tinggi di kuartil atas.
  - Jika secara bisnis (aturan bisnis platform) Cocokin wajib mengangkat kandidat Junior agar lebih kompetitif di lowongan berlabel "Entry Level", maka formula perhitungan overall_fit_score saat ini belum cukup. Tim Data Science perlu melakukan Hyperparameter Tuning. Misalnya, memberikan bobot afirmasi (bonus +0.15) secara khusus jika kandidat tanpa pengalaman melamar ke pekerjaan yang batas pengalamannya memang rendah (0-1 tahun), guna melawan bias organik dari kayanya kata kunci pada CV Senior.

## 7. Summary & Output Files

In [ ]:
import glob

print("📂 Output files di session storage:")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/*.csv")):
    size = os.path.getsize(f) / 1024
    print(f"   {os.path.basename(f):40s}  {size:7.1f} KB")

print(f"\n✅ Pipeline selesai!")
print(f"   A/B Test — Primary Test : {chosen_test}")
print(f"   Definitive p-value      : {final_p:.6e}")
print(f"   Cohen's d               : {cohen_d:.4f} ({eff_label})")
print(f"   Significant?            : {'YES ✅' if definitive else 'NO ❌'}")
print(f"   Observed Lift (B vs A)  : {lift:+.2f}%")

# Opsional: salin ke Google Drive
#
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# SAVE_TO = "/content/drive/MyDrive/CapstoneDicoding/outputs"
# os.makedirs(SAVE_TO, exist_ok=True)
# for f in glob.glob(f"{OUTPUT_DIR}/*.csv"):
#     shutil.copy(f, f"{SAVE_TO}/{os.path.basename(f)}")
# print(f"💾 Semua file disalin ke {SAVE_TO}")


# Kesimpulan Akhir

Analisis data secara keseluruhan membongkar fakta bahwa pasar kerja saat ini mengalami disrupsi ekspektasi yang menciptakan penghalang masuk (barrier to entry) yang sangat tidak adil bagi kandidat pemula (Fresh Graduate). Ketidakadilan ini dapat disimpulkan ke dalam tiga pilar penemuan utama:

1. Ilusi Posisi Entry-Level (Terjadi Inflasi Pengalaman)

 Label "Entry-Level" di pasar kerja saat ini hanyalah sebuah mitos. Perusahaan secara ekspektasi menuntut kandidat untuk memiliki rata-rata 3,26 tahun pengalaman sejak hari pertama bekerja. Hal ini menciptakan lingkaran setan di mana lulusan baru tidak bisa mendapatkan pekerjaan karena tidak punya pengalaman, dan tidak punya pengalaman karena tidak bisa mendapatkan pekerjaan.

2. Ketertinggalan Kepadatan Skill Secara Organik (Validasi A/B Test)

 Kekalahan kandidat Junior di pasar kerja bukan semata-mata karena bias HRD atau kurangnya durasi kerja, melainkan karena mereka secara kuantitatif tertinggal 24,32% dalam perbendaharaan keahlian (Skill Density) dibandingkan Senior. Eksperimen A/B Testing gagal mengangkat skor Junior karena algoritma membuktikan bahwa CV mereka secara organik lebih "miskin" kata kunci keahlian standar industri.

3. Kegagalan Keselarasan Akademis vs Industri Terapan

 Ketertinggalan 24% tersebut diperjelas oleh temuan missing skills. Dunia pendidikan terbukti belum mampu mengimbangi dinamika industri. Lulusan IT kehilangan skill spesifik level korporat (seperti DevOps dan Compliance), sementara banyak kandidat gagal memahami bahwa sektor tertentu (seperti Keuangan) menuntut portofolio Soft Skills yang jauh lebih tinggi (24,5%) dibandingkan sektor lainnya.

# Konklusi Startegis untuk produk Cocokin

  Ketiga pilar penemuan di atas memberikan landasan bisnis yang sangat solid: Platform pencari kerja tradisional hanya akan terus memenangkan kandidat Senior dan mengorbankan Junior. Oleh karena itu, platform Cocokin hadir dengan posisi yang unik, bukan sekadar sebagai mesin pencari lowongan, melainkan sebagai Katalisator Upskilling (Mentor Karier).

  Melalui fitur AI dan algoritma Sector-TF-IDF, Cocokin secara langsung mengintervensi masalah ini dengan cara:
   - Mendeteksi secara spesifik Missing Skills apa yang membuat kandidat tertinggal.
   - Memberikan rekomendasi pembelajaran yang terarah (misalnya: "Pelajari DevOps dan Javascript untuk sektor IT").
   - Membantu Fresh Graduate menambal ketertinggalan 24% tersebut, sehingga mereka dapat mem-bypass syarat pengalaman 3 tahun melalui pembuktian portofolio keahlian yang sangat padat dan relevan.

  Produk Cocokin terbukti sebagai solusi yang paling tepat sasaran untuk memecahkan kebekuan karier angkatan kerja muda saat ini.


